In [ ]:
# Outlier detection using Z-score method (|z| > 3)
from scipy import stats

print("Outlier Detection Summary (Z-score Method, |z| > 3):")
print("="*70)

for col in numerical_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    outliers_z = np.sum(z_scores > 3)
    outlier_percentage_z = (outliers_z / len(df)) * 100
    
    print(f"\n{col}:")
    print(f"  Outliers (Z-score > 3): {outliers_z} ({outlier_percentage_z:.2f}%)")

print("\n" + "="*70)
print("\nEDA Analysis Complete!")
print("\nKey Findings:")
print("- Review the distributions and histograms above")
print("- Check correlation heatmap for multicollinearity")
print("- Handle missing values appropriately")
print("- Investigate outliers before further processing")

In [ ]:
# Outlier detection using IQR method
print("Outlier Detection Summary (IQR Method):")
print("="*70)

outlier_summary = []
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df)) * 100
    
    outlier_summary.append({
        'Feature': col,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outlier Count': outlier_count,
        'Percentage': outlier_percentage
    })
    
    print(f"\n{col}:")
    print(f"  Q1: {Q1:.4f}, Q3: {Q3:.4f}, IQR: {IQR:.4f}")
    print(f"  Lower Bound: {lower_bound:.4f}, Upper Bound: {upper_bound:.4f}")
    print(f"  Outliers: {outlier_count} ({outlier_percentage:.2f}%)")

print("\n" + "="*70)

In [ ]:
# Create box plots for outlier detection
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    axes[idx].boxplot(df[col].dropna(), vert=True)
    axes[idx].set_title(f'Box Plot of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print("Box plot analysis completed.")

## 7. Outlier Detection

Use box plots and statistical methods (IQR, Z-score) to identify and visualize outliers in numerical features for further investigation.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

print("Missing Values Summary:")
print("="*50)
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Percentage': missing_percentage.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df) == 0:
    print("No missing values found!")
else:
    print(missing_df.to_string(index=False))

# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(12, 6))
    missing_data = df.isnull().sum().sort_values(ascending=False)
    missing_data = missing_data[missing_data > 0]
    
    plt.bar(range(len(missing_data)), missing_data.values, color='crimson', edgecolor='black')
    plt.xticks(range(len(missing_data)), missing_data.index, rotation=45, ha='right')
    plt.ylabel('Number of Missing Values')
    plt.title('Missing Values by Column', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to visualize.")

## 6. Identifying Missing Values

Identify missing values to determine missing data patterns and decide on appropriate imputation strategies.

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Display correlation matrix
print("Correlation Matrix:")
print(correlation_matrix)
print("\n")

# Create heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, 
            fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify highly correlated features
print("\nHighly Correlated Feature Pairs (|correlation| > 0.7):")
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            corr_pairs.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlation': correlation_matrix.iloc[i, j]
            })

if corr_pairs:
    corr_df = pd.DataFrame(corr_pairs)
    print(corr_df.to_string())
else:
    print("No highly correlated feature pairs found.")

## 5. Correlation Analysis

Understanding the relationship between numerical features to identify multicollinearity and feature associations.

In [ ]:
# Get categorical columns
categorical_cols = df.select_dtypes(include='object').columns

# Create bar plots for categorical features
n_cols = 2
n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))

if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[idx].bar(range(len(value_counts)), value_counts.values, color='coral', edgecolor='black')
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3, axis='y')

# Hide unused subplots
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print("Categorical distribution analysis completed.")

## 4. Distribution of Categorical Features

Analyzing the distribution of categorical features provides insights into the frequency and variability of categories.

In [ ]:
# Create KDE plots for numerical features
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    df[col].dropna().plot(kind='kde', ax=axes[idx], color='steelblue', linewidth=2)
    axes[idx].set_title(f'KDE Plot of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Density')
    axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print("KDE distribution analysis completed.")

In [ ]:
# Get numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns

# Create histograms for all numerical features
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col].dropna(), bins=30, color='skyblue', edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print("\nHistogram distribution analysis completed.")

## 3. Distribution of Numerical Features

Visualize the distribution of numerical features to identify patterns, skewness, and potential outliers.

In [ ]:
# Numerical Features Summary Statistics
print("Numerical Features Summary Statistics:")
print(df.describe().T)
print("\n" + "="*50 + "\n")

# Categorical Features Summary
print("Categorical Features Summary:")
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())

# Check for skewness
print("\n" + "="*50 + "\n")
print("Skewness of Numerical Features:")
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    skewness = df[col].skew()
    print(f"{col}: {skewness:.4f}")

## 2. Summary Statistics

Understanding the central tendency, dispersion, and shape of the dataset's distribution.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/data.csv')

# Display basic information
print("Dataset Shape:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("Data Types:")
print(df.dtypes)
print("\n" + "="*50 + "\n")

print("Dataset Info:")
df.info()
print("\n" + "="*50 + "\n")

print("First Few Rows:")
df.head(10)

## 1. Overview of the Data

Understanding the structure of the dataset, including the number of rows, columns, and data types.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Exploratory Data Analysis (EDA)

This notebook explores the dataset to uncover patterns, identify data quality issues, and form hypotheses to guide feature engineering for the credit risk model.

**Objective:** 
- Understand data structure and quality
- Identify missing values and outliers
- Discover patterns and relationships in features
- Generate insights for feature engineering